# Predicting World Cup Success

This notebook summarizes the analytical workflow behind the project. The production pipeline lives in `src/` and `sql/`; this notebook is meant for portfolio review and explanation.

## Research Question

Which measurable factors are most associated with deep FIFA Men's World Cup runs: goal difference, defense, attack, host advantage, tournament experience, confederation, knockout efficiency, and pre-tournament form?

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from clean_data import build_processed_data
from run_sql_analysis import run_sql_scripts

build_processed_data()
run_sql_scripts()

features = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'team_tournament_features.csv')
features.head()

## Data Integrity Notes

- Raw match data comes from public CSV files.
- Team names are standardized across World Cup and international-results sources.
- Goals are validated as numeric and non-negative.
- Historical experience is calculated only from previous tournaments.
- Pre-tournament form excludes FIFA World Cup matches to avoid leakage.

In [ ]:
con = duckdb.connect(str(PROJECT_ROOT / 'data' / 'processed' / 'world_cup_success.duckdb'))
con.execute('SELECT * FROM factor_correlations').df()

In [ ]:
con.execute('''
SELECT
    stage_reached,
    team_tournaments,
    ROUND(avg_goal_difference, 2) AS avg_goal_difference,
    ROUND(avg_goals_against_per_match, 2) AS avg_goals_against_per_match,
    ROUND(avg_pre_wc_points_per_match, 2) AS avg_pre_wc_points_per_match
FROM stage_factor_summary
ORDER BY stage_score
''').df()

## Finding 1: Goal Difference Separates Deep Runs

Goal difference has the strongest descriptive relationship with stage reached in this dataset. This does not prove causation, but it is the clearest observable pattern: teams that go deep usually create separation across the whole tournament, not only in one match.

In [ ]:
con.execute('''
SELECT *
FROM host_advantage_summary
ORDER BY is_host DESC
''').df()

## Finding 2: Host Advantage Is Visible, But Not Causal Proof

Host teams have advanced deeper on average. The analysis treats this as an association because host performance can reflect several factors at once: crowd support, travel, familiarity, qualification path, and the quality of the host team.

In [ ]:
con.execute('''
SELECT team_name, group_name, confederation_code, prior_world_cups, prior_titles, pre_wc_points_per_match
FROM preview_2026_ranked
ORDER BY readiness_score DESC
LIMIT 12
''').df()

## 2026 Preview

The 2026 table verifies the expanded 48-team field and groups. It is a preview based on historical experience and recent form only. It does not include any 2026 World Cup match results.